# Train SISO Thermal Model from ESP32 Telemetry

This notebook trains a supervised ML system-identification model for a SISO thermal plant. The control input is `heater_pwm`, the measured output is `temp_c`, and pump-related telemetry is treated as a measured disturbance. The model predicts future temperature deltas for ML-MPC use; it is not direct reinforcement learning and it should not bypass ESP32 safety logic such as `hard_kill`, `manual_kill`, or `heater_lockout`.

In [ ]:
import json
import time
from contextlib import contextmanager
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_PATH = "esp32_telemetry.csv"
MODEL_DIR = Path("models")
HORIZONS_S = [1, 5, 10]

# CUDA acceleration is optional. sklearn models remain CPU-only, so the notebook
# adds GPU candidates when XGBoost GPU or RAPIDS cuML are installed.
USE_CUDA = True
INCLUDE_CPU_BASELINES = True
USE_NSIGHT_NVTX = True

pd.set_option("display.max_columns", 120)
plt.rcParams["figure.figsize"] = (12, 5)

## CUDA and Nsight Setup

The default sklearn estimators do not train on CUDA. When available, this notebook adds GPU-backed model candidates from XGBoost and RAPIDS cuML, downcasts training arrays to `float32`, and wraps training/prediction blocks in NVTX ranges for Nsight Systems. Install one GPU backend such as `xgboost` with CUDA support or RAPIDS `cuml`, plus optional `nvtx` for named ranges. To profile the notebook end-to-end, run it under Nsight Systems, for example: `nsys profile -t cuda,nvtx,osrt,cublas -o thermal_training python -m jupyter nbconvert --to notebook --execute train_siso_thermal_model.ipynb`. Use Nsight Compute only after Nsight Systems identifies a specific CUDA kernel worth drilling into.

In [ ]:
try:
    import nvtx
    HAS_NVTX = True
except ImportError:
    nvtx = None
    HAS_NVTX = False

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    XGBRegressor = None
    HAS_XGBOOST = False

try:
    from cuml.ensemble import RandomForestRegressor as CuMLRandomForestRegressor
    HAS_CUML = True
except ImportError:
    CuMLRandomForestRegressor = None
    HAS_CUML = False


@contextmanager
def profile_range(name, color="blue"):
    if USE_NSIGHT_NVTX and HAS_NVTX:
        with nvtx.annotate(name, color=color):
            yield
    else:
        yield


def to_numpy_1d(values):
    if hasattr(values, "to_numpy"):
        values = values.to_numpy()
    try:
        import cupy as cp
        if isinstance(values, cp.ndarray):
            values = cp.asnumpy(values)
    except ImportError:
        pass
    return np.asarray(values).reshape(-1)


print(f"NVTX annotations enabled: {HAS_NVTX and USE_NSIGHT_NVTX}")
print(f"XGBoost available: {HAS_XGBOOST}")
print(f"RAPIDS cuML available: {HAS_CUML}")
if USE_CUDA and not (HAS_XGBOOST or HAS_CUML):
    print("CUDA requested, but no supported GPU training backend was imported. Falling back to sklearn CPU models.")

## Load and Clean Telemetry

The ESP32 logger emits numeric CSV telemetry. The cleaning stage keeps the required plant-identification fields, removes hard-kill rows from training data, and fills missing optional disturbance/safety columns with conservative defaults so older CSVs remain usable.

In [ ]:
REQUIRED_COLUMNS = ["temp_c", "ms", "heater_pwm", "pump_on"]

OPTIONAL_DEFAULTS = {
    "event": 0,
    "adc": np.nan,
    "dtemp_c_per_s": 0.0,
    "setpoint_c": np.nan,
    "mode": 0,
    "heating": 0,
    "heater_lockout": 0,
    "pump_enabled": 0,
    "pump_allowed": 0,
    "motor_pwm": 0,
    "motor_on_ms": 0,
    "motor_period_ms": 0,
    "temp_before_pump_c": np.nan,
    "min_temp_after_pump_c": np.nan,
    "last_pump_drop_c": 0.0,
    "recovery_time_s": 0.0,
    "manual_kill": 0,
    "hard_kill": 0,
    "uptime_s": np.nan,
}


def load_and_clean_telemetry(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"CSV file not found: {path.resolve()}")

    df = pd.read_csv(path)
    df.columns = [str(col).strip() for col in df.columns]

    missing_required = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    for col, default in OPTIONAL_DEFAULTS.items():
        if col not in df.columns:
            df[col] = default

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    before = len(df)
    df = df.dropna(subset=REQUIRED_COLUMNS).copy()
    dropped_required = before - len(df)

    df["hard_kill"] = df["hard_kill"].fillna(0)
    hard_kill_rows = int((df["hard_kill"] == 1).sum())
    df = df[df["hard_kill"] != 1].copy()

    df = df.sort_values("ms").drop_duplicates(subset="ms", keep="first").reset_index(drop=True)
    df["time_s"] = df["ms"] / 1000.0

    df["heater_pwm"] = df["heater_pwm"].clip(0, 255)
    df["motor_pwm"] = df["motor_pwm"].fillna(0).clip(0, 255)
    df["pump_on"] = df["pump_on"].fillna(0).clip(0, 1)
    df["event"] = df["event"].fillna(0).astype(int)
    df["heater_lockout"] = df["heater_lockout"].fillna(0).clip(0, 1)

    if df["setpoint_c"].isna().all():
        df["setpoint_c"] = df["temp_c"]
    else:
        df["setpoint_c"] = df["setpoint_c"].ffill().bfill().fillna(df["temp_c"])

    if df["dtemp_c_per_s"].isna().all():
        dt = df["time_s"].diff().replace(0, np.nan)
        df["dtemp_c_per_s"] = df["temp_c"].diff() / dt
    df["dtemp_c_per_s"] = df["dtemp_c_per_s"].replace([np.inf, -np.inf], np.nan).fillna(0)

    if not df["time_s"].is_monotonic_increasing:
        raise ValueError("time_s is not monotonic after sorting; check ms values")

    print(f"Loaded rows: {before}")
    print(f"Dropped rows missing required fields: {dropped_required}")
    print(f"Removed hard_kill rows: {hard_kill_rows}")
    print(f"Rows available after cleaning: {len(df)}")
    print(f"Time span: {df['time_s'].min():.2f}s to {df['time_s'].max():.2f}s")
    print(f"Temp range: {df['temp_c'].min():.2f}C to {df['temp_c'].max():.2f}C")
    print(f"Heater PWM range: {df['heater_pwm'].min():.0f} to {df['heater_pwm'].max():.0f}")
    print(f"Pump-on rows: {int((df['pump_on'] == 1).sum())}")
    print("Event counts:")
    print(df["event"].value_counts(dropna=False).sort_index())
    display(df.head())
    display(df.describe().T[["count", "mean", "std", "min", "max"]])
    return df


df_clean = load_and_clean_telemetry(DATA_PATH)

## Feature Engineering

The feature set includes lagged plant state, lagged control, rolling thermal behavior, disturbance indicators, and piecewise operating-mode flags. These are intended for supervised dynamics learning: predict how temperature changes over a fixed horizon given the current state and control.

In [ ]:
def add_features(df):
    out = df.copy()

    lag_specs = {
        "temp_c_lag_1": ("temp_c", 1),
        "temp_c_lag_2": ("temp_c", 2),
        "temp_c_lag_5": ("temp_c", 5),
        "heater_pwm_lag_1": ("heater_pwm", 1),
        "heater_pwm_lag_2": ("heater_pwm", 2),
        "pump_on_lag_1": ("pump_on", 1),
        "dtemp_c_per_s_lag_1": ("dtemp_c_per_s", 1),
    }
    for new_col, (base_col, lag) in lag_specs.items():
        out[new_col] = out[base_col].shift(lag)

    out["temp_mean_5"] = out["temp_c"].rolling(window=5, min_periods=1).mean()
    out["temp_std_5"] = out["temp_c"].rolling(window=5, min_periods=2).std().fillna(0)
    out["pwm_mean_5"] = out["heater_pwm"].rolling(window=5, min_periods=1).mean()
    out["pump_recent_10"] = out["pump_on"].rolling(window=10, min_periods=1).max()

    out["temp_error"] = out["setpoint_c"] - out["temp_c"]
    out["pwm_delta"] = out["heater_pwm"] - out["heater_pwm_lag_1"]
    out["pump_event_active"] = out["event"].isin([1, 2, 3]).astype(int)

    out["mode_normal"] = ((out["event"] == 0) & (out["pump_on"] == 0)).astype(int)
    out["mode_pump_start"] = (out["event"] == 1).astype(int)
    out["mode_pump_running"] = (out["pump_on"] == 1).astype(int)
    out["mode_recovery"] = (out["event"] == 3).astype(int)
    out["mode_lockout"] = (out["heater_lockout"] == 1).astype(int)

    return out


df_features = add_features(df_clean)
display(df_features.head(10))

## Time-Based Future Targets

The target builder uses nearest-time matching instead of row offsets, so it remains valid when the sampling period is not perfectly constant. Models are trained to predict future temperature deltas, then reconstructed as `current temp_c + predicted delta`.

In [ ]:
def add_time_based_targets(df, horizons_s):
    out = df.sort_values("time_s").reset_index(drop=True).copy()
    lookup = out[["time_s", "temp_c"]].rename(columns={"time_s": "lookup_time_s", "temp_c": "matched_temp_c"})

    for horizon in horizons_s:
        target_query = pd.DataFrame({"target_time_s": out["time_s"] + horizon})
        matched = pd.merge_asof(
            target_query.sort_values("target_time_s"),
            lookup.sort_values("lookup_time_s"),
            left_on="target_time_s",
            right_on="lookup_time_s",
            direction="nearest",
        )
        time_error = (matched["lookup_time_s"] - matched["target_time_s"]).abs()
        target_temp_col = f"target_temp_{horizon}s"
        target_delta_col = f"target_dtemp_{horizon}s"
        match_error_col = f"target_match_error_{horizon}s"
        out[target_temp_col] = matched["matched_temp_c"].to_numpy()
        out[target_delta_col] = out[target_temp_col] - out["temp_c"]
        out[match_error_col] = time_error.to_numpy()

    return out


df_model = add_time_based_targets(df_features, HORIZONS_S)

base_feature_columns = [
    "temp_c",
    "heater_pwm",
    "pump_on",
    "event",
    "motor_pwm",
    "dtemp_c_per_s",
    "setpoint_c",
    "heater_lockout",
    "temp_c_lag_1",
    "temp_c_lag_2",
    "temp_c_lag_5",
    "heater_pwm_lag_1",
    "heater_pwm_lag_2",
    "pump_on_lag_1",
    "dtemp_c_per_s_lag_1",
    "temp_mean_5",
    "temp_std_5",
    "pwm_mean_5",
    "pump_recent_10",
    "temp_error",
    "pwm_delta",
    "pump_event_active",
    "mode_normal",
    "mode_pump_start",
    "mode_pump_running",
    "mode_recovery",
    "mode_lockout",
]

target_columns = [f"target_dtemp_{h}s" for h in HORIZONS_S]
model_columns = base_feature_columns + target_columns + [f"target_temp_{h}s" for h in HORIZONS_S]
df_model_ready = df_model.dropna(subset=model_columns).reset_index(drop=True)

print(f"Rows before dropping lag/target NaNs: {len(df_model)}")
print(f"Rows ready for modeling: {len(df_model_ready)}")
for horizon in HORIZONS_S:
    print(
        f"Horizon {horizon}s nearest-match error: "
        f"median={df_model_ready[f'target_match_error_{horizon}s'].median():.3f}s, "
        f"p95={df_model_ready[f'target_match_error_{horizon}s'].quantile(0.95):.3f}s"
    )

display(df_model_ready[["time_s", "temp_c", "heater_pwm"] + target_columns].head())

## Chronological Train / Validation / Test Split

Thermal telemetry is time series data, so the split is chronological: first 70% for training, next 15% for validation, and final 15% for test.

In [ ]:
def chronological_split(df, train_frac=0.70, val_frac=0.15):
    n = len(df)
    if n < 30:
        raise ValueError(f"Need more rows for chronological train/validation/test split, got {n}")
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    return train_df, val_df, test_df


train_df, val_df, test_df = chronological_split(df_model_ready)
print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Train time: {train_df['time_s'].min():.2f}s to {train_df['time_s'].max():.2f}s")
print(f"Validation time: {val_df['time_s'].min():.2f}s to {val_df['time_s'].max():.2f}s")
print(f"Test time: {test_df['time_s'].min():.2f}s to {test_df['time_s'].max():.2f}s")

## Train Candidate Models

Each horizon gets its own regression model because the temperature response shape changes with prediction distance. Model selection is based on validation RMSE for delta temperature. GPU candidates are tried first when available; sklearn baselines remain available for comparison and fallback.

In [ ]:
def build_candidate_models():
    candidates = {}

    if USE_CUDA and HAS_XGBOOST:
        # XGBoost 2.x uses device='cuda'. Older versions generally ignore unknown
        # sklearn-wrapper kwargs or emit a warning; if fitting fails, the training
        # loop below skips this candidate and continues with the next model.
        candidates["XGBRegressorCUDA"] = {
            "backend": "cuda",
            "model": XGBRegressor(
                n_estimators=600,
                max_depth=4,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="reg:squarederror",
                tree_method="hist",
                device="cuda",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        }

    if USE_CUDA and HAS_CUML:
        candidates["CuMLRandomForestRegressor"] = {
            "backend": "cuda",
            "model": CuMLRandomForestRegressor(
                n_estimators=300,
                max_depth=16,
                n_streams=4,
                random_state=RANDOM_STATE,
            ),
        }

    if INCLUDE_CPU_BASELINES or not candidates:
        candidates.update({
            "RandomForestRegressor": {
                "backend": "cpu",
                "model": RandomForestRegressor(
                    n_estimators=200,
                    min_samples_leaf=3,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            },
            "GradientBoostingRegressor": {
                "backend": "cpu",
                "model": GradientBoostingRegressor(random_state=RANDOM_STATE),
            },
            "HistGradientBoostingRegressor": {
                "backend": "cpu",
                "model": HistGradientBoostingRegressor(
                    random_state=RANDOM_STATE,
                    max_iter=250,
                    learning_rate=0.05,
                ),
            },
        })

    return candidates


def regression_metrics(y_true, y_pred):
    errors = y_pred - y_true
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mse)),
        "R2": r2_score(y_true, y_pred),
        "within_0p5C_percent": 100.0 * np.mean(np.abs(errors) <= 0.5),
        "within_1C_percent": 100.0 * np.mean(np.abs(errors) <= 1.0),
        "within_2C_percent": 100.0 * np.mean(np.abs(errors) <= 2.0),
    }


# float32 reduces memory traffic and is the preferred dtype for most GPU trainers.
X_train = train_df[base_feature_columns].astype(np.float32)
X_val = val_df[base_feature_columns].astype(np.float32)
X_test = test_df[base_feature_columns].astype(np.float32)

best_models = {}
model_results = []

for horizon in HORIZONS_S:
    target_col = f"target_dtemp_{horizon}s"
    y_train = train_df[target_col].astype(np.float32)
    y_val = val_df[target_col].astype(np.float32)

    print(f"\nTraining horizon {horizon}s")
    candidates = build_candidate_models()
    best_name = None
    best_model = None
    best_backend = None
    best_rmse = np.inf

    for name, candidate in candidates.items():
        model = candidate["model"]
        backend = candidate["backend"]
        started = time.perf_counter()
        try:
            with profile_range(f"fit_h{horizon}s_{name}", color="green"):
                model.fit(X_train, y_train)
            with profile_range(f"predict_val_h{horizon}s_{name}", color="purple"):
                val_pred = to_numpy_1d(model.predict(X_val))
        except Exception as exc:
            elapsed_s = time.perf_counter() - started
            print(f"  Skipping {name} ({backend}) after {elapsed_s:.2f}s: {exc}")
            model_results.append({
                "horizon_s": horizon,
                "model": name,
                "backend": backend,
                "split": "validation",
                "fit_time_s": elapsed_s,
                "status": "failed",
                "error": str(exc),
            })
            continue

        elapsed_s = time.perf_counter() - started
        metrics = regression_metrics(y_val, val_pred)
        model_results.append({
            "horizon_s": horizon,
            "model": name,
            "backend": backend,
            "split": "validation",
            "fit_time_s": elapsed_s,
            "status": "ok",
            "error": "",
            **metrics,
        })
        print(
            f"  {name} [{backend}]: val RMSE={metrics['RMSE']:.4f}, "
            f"MAE={metrics['MAE']:.4f}, R2={metrics['R2']:.4f}, time={elapsed_s:.2f}s"
        )
        if metrics["RMSE"] < best_rmse:
            best_name = name
            best_model = model
            best_backend = backend
            best_rmse = metrics["RMSE"]

    if best_model is None:
        raise RuntimeError(f"No model successfully trained for {horizon}s horizon")

    best_models[horizon] = {"name": best_name, "backend": best_backend, "model": best_model}
    print(f"Selected {best_name} [{best_backend}] for {horizon}s horizon")

validation_results_df = pd.DataFrame(model_results)
display(validation_results_df.sort_values(["horizon_s", "RMSE"]))

## Test Metrics and Piecewise Mode Evaluation

Final metrics are reported on the held-out chronological test segment. Piecewise evaluation checks whether model error changes when the pump is off, pump is on, recovery is active, or heater lockout is active.

In [ ]:
def evaluate_by_mode(df, y_true, y_pred):
    mode_masks = {
        "pump_off": df["pump_on"] == 0,
        "pump_on": df["pump_on"] == 1,
        "recovery": df["mode_recovery"] == 1,
        "lockout": df["mode_lockout"] == 1,
    }
    rows = []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    for mode_name, mask in mode_masks.items():
        idx = np.asarray(mask)
        if idx.sum() == 0:
            rows.append({"mode": mode_name, "n": 0, "MAE": np.nan, "RMSE": np.nan})
            continue
        rows.append({
            "mode": mode_name,
            "n": int(idx.sum()),
            "MAE": mean_absolute_error(y_true[idx], y_pred[idx]),
            "RMSE": float(np.sqrt(mean_squared_error(y_true[idx], y_pred[idx]))),
        })
    return pd.DataFrame(rows)


test_predictions = {}
metrics_rows = []
mode_eval_rows = []

for horizon in HORIZONS_S:
    target_col = f"target_dtemp_{horizon}s"
    y_test = test_df[target_col]
    selected = best_models[horizon]
    with profile_range(f"predict_test_h{horizon}s_{selected['name']}", color="orange"):
        pred_delta = to_numpy_1d(selected["model"].predict(X_test))
    test_predictions[horizon] = pred_delta

    metrics = regression_metrics(y_test, pred_delta)
    metrics_rows.append({
        "horizon_s": horizon,
        "selected_model": selected["name"],
        "backend": selected["backend"],
        "split": "test",
        **metrics,
    })

    mode_eval = evaluate_by_mode(test_df, y_test, pred_delta)
    mode_eval["horizon_s"] = horizon
    mode_eval["selected_model"] = selected["name"]
    mode_eval["backend"] = selected["backend"]
    mode_eval_rows.append(mode_eval)

metrics_summary_df = pd.DataFrame(metrics_rows)
mode_eval_df = pd.concat(mode_eval_rows, ignore_index=True)

print("Test metrics")
display(metrics_summary_df)
print("Piecewise mode evaluation")
display(mode_eval_df[["horizon_s", "selected_model", "backend", "mode", "n", "MAE", "RMSE"]])

## Diagnostic Plots

These plots compare predicted and actual delta temperature, reconstructed future temperature, residuals over time, and feature importance where the selected model exposes importance values.

In [ ]:
def plot_horizon_diagnostics(horizon):
    target_delta_col = f"target_dtemp_{horizon}s"
    target_temp_col = f"target_temp_{horizon}s"
    model_info = best_models[horizon]
    model = model_info["model"]
    pred_delta = test_predictions[horizon]
    actual_delta = test_df[target_delta_col].to_numpy()
    pred_future_temp = test_df["temp_c"].to_numpy() + pred_delta
    actual_future_temp = test_df[target_temp_col].to_numpy()
    residual = pred_delta - actual_delta
    time_s = test_df["time_s"].to_numpy()

    plt.figure()
    plt.plot(time_s, actual_delta, label="Actual delta temp")
    plt.plot(time_s, pred_delta, label="Predicted delta temp", alpha=0.85)
    plt.title(f"{horizon}s Horizon: Actual vs Predicted Delta Temperature")
    plt.xlabel("time_s")
    plt.ylabel("delta temp_c")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure()
    plt.plot(time_s, actual_future_temp, label="Actual future temp")
    plt.plot(time_s, pred_future_temp, label="Predicted future temp", alpha=0.85)
    plt.title(f"{horizon}s Horizon: Reconstructed Future Temperature")
    plt.xlabel("time_s")
    plt.ylabel("temp_c")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure()
    plt.plot(time_s, residual, label="Residual")
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"{horizon}s Horizon: Residual Over Time")
    plt.xlabel("time_s")
    plt.ylabel("predicted - actual delta temp_c")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    if hasattr(model, "feature_importances_"):
        importance = pd.Series(model.feature_importances_, index=base_feature_columns).sort_values(ascending=False)
        plt.figure(figsize=(12, 7))
        importance.head(20).iloc[::-1].plot(kind="barh")
        plt.title(f"{horizon}s Horizon: Top Feature Importances ({model_info['name']})")
        plt.xlabel("importance")
        plt.grid(True, axis="x", alpha=0.3)
        plt.show()
    else:
        print(f"{model_info['name']} does not expose feature_importances_ for {horizon}s horizon")


for horizon in HORIZONS_S:
    plot_horizon_diagnostics(horizon)

## Save Model Artifacts

The saved artifacts are intentionally simple: one model per horizon, the exact feature-column order needed for inference, a CSV of final test metrics, and lightweight metadata recording which backend won for each horizon.

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)

for horizon in HORIZONS_S:
    joblib.dump(best_models[horizon]["model"], MODEL_DIR / f"best_model_{horizon}s.joblib")

with open(MODEL_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(base_feature_columns, f, indent=2)

metrics_summary_df.to_csv(MODEL_DIR / "metrics_summary.csv", index=False)

model_metadata = {
    "use_cuda_requested": USE_CUDA,
    "nvtx_enabled": bool(USE_NSIGHT_NVTX and HAS_NVTX),
    "selected_models": {
        f"{horizon}s": {
            "name": best_models[horizon]["name"],
            "backend": best_models[horizon]["backend"],
        }
        for horizon in HORIZONS_S
    },
}
with open(MODEL_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

print(f"Saved artifacts to {MODEL_DIR.resolve()}")
print("Saved:")
for path in sorted(MODEL_DIR.iterdir()):
    print(f"  {path.name}")

## MPC-Ready Prediction Function

This function loads a saved model, builds a one-row feature frame in the saved training order, predicts delta temperature, and returns reconstructed future `temp_c`. For live deployment, keep ESP32 safety logic separate; do not allow this model to override `hard_kill`, `manual_kill`, or `heater_lockout` interlocks.

In [ ]:
def _feature_value(row_dict, key, default=0.0):
    value = row_dict.get(key, default)
    if value is None or pd.isna(value):
        return default
    return value


def build_feature_row_from_state(row_dict, feature_columns):
    row = dict(row_dict)
    row.setdefault("event", 0)
    row.setdefault("pump_on", 0)
    row.setdefault("motor_pwm", 0)
    row.setdefault("heater_lockout", 0)
    row.setdefault("dtemp_c_per_s", 0)
    row.setdefault("setpoint_c", row.get("temp_c", 0))

    # For single-row live inference, lag/rolling values should ideally come from a telemetry buffer.
    # If they are absent, fall back to the current state so inference remains well-defined.
    row.setdefault("temp_c_lag_1", row.get("temp_c", 0))
    row.setdefault("temp_c_lag_2", row.get("temp_c_lag_1", row.get("temp_c", 0)))
    row.setdefault("temp_c_lag_5", row.get("temp_c_lag_2", row.get("temp_c", 0)))
    row.setdefault("heater_pwm_lag_1", row.get("heater_pwm", 0))
    row.setdefault("heater_pwm_lag_2", row.get("heater_pwm_lag_1", row.get("heater_pwm", 0)))
    row.setdefault("pump_on_lag_1", row.get("pump_on", 0))
    row.setdefault("dtemp_c_per_s_lag_1", row.get("dtemp_c_per_s", 0))
    row.setdefault("temp_mean_5", row.get("temp_c", 0))
    row.setdefault("temp_std_5", 0)
    row.setdefault("pwm_mean_5", row.get("heater_pwm", 0))
    row.setdefault("pump_recent_10", row.get("pump_on", 0))

    row["temp_error"] = row.get("setpoint_c", row.get("temp_c", 0)) - row.get("temp_c", 0)
    row["pwm_delta"] = row.get("heater_pwm", 0) - row.get("heater_pwm_lag_1", row.get("heater_pwm", 0))
    row["pump_event_active"] = int(row.get("event", 0) in [1, 2, 3])
    row["mode_normal"] = int((row.get("event", 0) == 0) and (row.get("pump_on", 0) == 0))
    row["mode_pump_start"] = int(row.get("event", 0) == 1)
    row["mode_pump_running"] = int(row.get("pump_on", 0) == 1)
    row["mode_recovery"] = int(row.get("event", 0) == 3)
    row["mode_lockout"] = int(row.get("heater_lockout", 0) == 1)

    return pd.DataFrame([{col: _feature_value(row, col, 0.0) for col in feature_columns}]).astype(np.float32)


def predict_future_temperature(row_dict, horizon_s):
    '''
    Takes one telemetry state dictionary and horizon_s in [1, 5, 10].
    Returns predicted future temp_c.
    '''
    if horizon_s not in [1, 5, 10]:
        raise ValueError("horizon_s must be one of [1, 5, 10]")

    with open(MODEL_DIR / "feature_columns.json", "r", encoding="utf-8") as f:
        feature_columns = json.load(f)

    model = joblib.load(MODEL_DIR / f"best_model_{horizon_s}s.joblib")
    X_one = build_feature_row_from_state(row_dict, feature_columns)
    predicted_delta = float(to_numpy_1d(model.predict(X_one))[0])
    return float(row_dict["temp_c"] + predicted_delta)


example_state = test_df.iloc[0][base_feature_columns].to_dict()
print("Example 5s predicted future temp_c:", predict_future_temperature(example_state, 5))

## Offline ML-MPC Demo

This is an offline receding-horizon demonstration using the trained 5-second model. It evaluates candidate PWM actions against a simple cost function. The output is a recommendation trace only; embedded safety constraints remain separate and authoritative.

In [ ]:
candidate_pwms = [0, 64, 128, 184, 220, 255]
demo_len = min(250, len(test_df))
demo_df = test_df.iloc[:demo_len].copy().reset_index(drop=True)
mpc_model = best_models[5]["model"]

recommended_pwms = []
predicted_temps = []
previous_pwm = float(demo_df.loc[0, "heater_pwm"] if len(demo_df) else 0)

for _, row in demo_df.iterrows():
    safety_active = any([
        row.get("manual_kill", 0) == 1,
        row.get("hard_kill", 0) == 1,
        row.get("heater_lockout", 0) == 1,
    ])
    if safety_active:
        recommended_pwms.append(0)
        predicted_temps.append(float(row["temp_c"]))
        previous_pwm = 0.0
        continue

    best_pwm = None
    best_cost = np.inf
    best_predicted_temp = np.nan

    for pwm in candidate_pwms:
        candidate_state = row[base_feature_columns].to_dict()
        candidate_state["heater_pwm"] = pwm
        candidate_state["pwm_delta"] = pwm - candidate_state.get("heater_pwm_lag_1", previous_pwm)
        candidate_state["pwm_mean_5"] = np.mean([candidate_state.get("pwm_mean_5", pwm), pwm])

        X_candidate = build_feature_row_from_state(candidate_state, base_feature_columns)
        predicted_delta = float(to_numpy_1d(mpc_model.predict(X_candidate))[0])
        predicted_temp = float(candidate_state["temp_c"] + predicted_delta)
        setpoint_c = float(candidate_state.get("setpoint_c", candidate_state["temp_c"]))
        cost = (predicted_temp - setpoint_c) ** 2 + 0.0005 * pwm ** 2 + 0.05 * (pwm - previous_pwm) ** 2

        if cost < best_cost:
            best_cost = cost
            best_pwm = pwm
            best_predicted_temp = predicted_temp

    recommended_pwms.append(best_pwm)
    predicted_temps.append(best_predicted_temp)
    previous_pwm = float(best_pwm)

demo_df["recommended_pwm"] = recommended_pwms
demo_df["mpc_predicted_temp_5s"] = predicted_temps

plt.figure()
plt.step(demo_df["time_s"], demo_df["heater_pwm"], where="post", label="Actual heater_pwm")
plt.step(demo_df["time_s"], demo_df["recommended_pwm"], where="post", label="Recommended PWM", alpha=0.85)
plt.title("Offline MPC Demo: Actual vs Recommended PWM")
plt.xlabel("time_s")
plt.ylabel("heater_pwm")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure()
plt.plot(demo_df["time_s"], demo_df["temp_c"], label="temp_c")
plt.plot(demo_df["time_s"], demo_df["setpoint_c"], label="setpoint_c")
plt.plot(demo_df["time_s"], demo_df["mpc_predicted_temp_5s"], label="Predicted temp after recommended PWM", alpha=0.75)
plt.title("Offline MPC Demo: Temperature and Setpoint")
plt.xlabel("time_s")
plt.ylabel("temp_c")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

display(demo_df[["time_s", "temp_c", "setpoint_c", "heater_pwm", "recommended_pwm", "mpc_predicted_temp_5s"]].head(20))